# formal_30_v1 / formal_30_v2 正式实验结果汇总

本 Notebook 只使用两个冻结快照：

- `formal_30_v1_results_20261015_161053`：30 个独立测试样本的自动迁移基线。
- `formal_30_v2_manual_results_20261022_152408`：14 个定向样本的事后人工浏览器诊断与修复。

**统计边界：** `formal_30_v1` 可以用于报告无偏自动迁移结果；`formal_30_v2` 的 14/14 是人工介入后的复验结果，不能表述为 30 个样本的自动迁移成功率。

In [ ]:
from pathlib import Path
import csv
import json
import subprocess
import sys

cwd = Path.cwd().resolve()
ROOT = next(path for path in (cwd, *cwd.parents) if (path / 'migration_pipeline').is_dir())
subprocess.run([sys.executable, str(ROOT / 'migration_pipeline/analyze_formal_30_v1_v2.py')], cwd=ROOT, check=True)
ANALYSIS_DIR = ROOT / 'data/experiments/analysis/formal_30_v1_v2_20261022'
summary = json.loads((ANALYSIS_DIR / 'summary.json').read_text(encoding='utf-8'))
with (ANALYSIS_DIR / 'sample_results.csv').open(encoding='utf-8-sig', newline='') as handle:
    sample_rows = list(csv.DictReader(handle))
assert summary['formal_30_v1']['aggregate']['configured'] == 30
assert summary['formal_30_v2']['registered'] == 14
assert summary['formal_30_v2']['manual_passed'] == 14
summary['sources']

## 1. formal_30_v1 自动迁移基线

| 指标 | 结果 |
| --- | ---: |
| 配置样本数 | 30 |
| 正常完成 | 29 |
| 自动通过 | 16 |
| 自动失败 | 13 |
| 执行异常 | 1 |
| 完成样本通过率 | 55.17% |
| 全部配置样本通过率 | 53.33% |
| 初次通过 | 3 |
| Repair 后转为通过 | 13 |
| Repair 总轮数 | 51 |
| 平均 Repair 轮数 | 1.759 |
| Generation Token | 701,340 |
| Repair Token | 1,767,836 |
| 总 Token | 2,469,176 |

自动基线应优先报告 `16/30 = 53.33%`，因为它把执行异常也保留在完整实验分母中；`16/29 = 55.17%` 可作为正常完成样本的补充口径。

## 2. 复杂度分层

| 复杂度 | 配置 | 完成 | 通过 | 失败 | 异常 | 完成样本通过率 | 全部配置通过率 |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| simple | 10 | 10 | 9 | 1 | 0 | 90.00% | 90.00% |
| medium | 10 | 10 | 5 | 5 | 0 | 50.00% | 50.00% |
| complex | 10 | 9 | 2 | 7 | 1 | 22.22% | 20.00% |

复杂度与自动通过率呈明显负相关：从 simple 的 90.00% 降至 complex 的 20.00%。复杂组件中的生命周期、嵌套状态、动态样式和多步骤交互更容易暴露框架语义差异。

In [ ]:
aggregate = summary['formal_30_v1']['aggregate']
for row in summary['formal_30_v1']['complexity']:
    configured_rate = row['pass_rate_configured'] * 100
    print(f"{row['complexity']:7s}: {row['passed']}/{row['configured']} = {configured_rate:.2f}%")

## 3. formal_30_v2 人工复核总体结果

- 登记并完成人工浏览器复核：14/14。
- 产生 v2 修复代码：11 个样本。
- 无须修改、人工确认通过：3 个样本。
- 这 14 个样本在 v1 中的结果分布：5 个自动通过、8 个自动失败、1 个执行异常。
- 自动通过但人工发现浏览器缺陷：`DeliveryPhaseCard`、`EnergyLoadScheduler`。

这两个自动通过后的人工缺陷说明当前静态校验和轻量 DOM 比较存在盲点，但由于 14 个样本是定向复核而非随机抽样，不能据此估计全部自动通过样本的假阳性率。

## 4. 14 个样本明细

| 样本 | 复杂度 | v1 自动结果 | v2 修改 | 主类别 | 人工结果 | 主要现象 |
| --- | --- | --- | --- | --- | --- | --- |
| DeliveryPhaseCard | simple | passed | 是 | 循环作用域绑定 | passed | 阶段推进后颜色不随活动阶段变化 |
| PollResults | medium | passed | 否 | 未发现差异 | passed | 人工浏览器检查未发现差异 |
| ImageCropControls | medium | failed | 是 | 初始化或生命周期迁移 | passed | 缺少初始值，参数、画布和宽高比锁定异常 |
| AttendanceRoster | medium | passed | 否 | 未发现差异 | passed | 人工浏览器检查未发现差异 |
| InventoryBatchManager | complex | failed | 是 | 模板隐藏响应式依赖 | passed | 单行选择和全选后样式不更新 |
| MediaReviewStudio | complex | failed | 是 | 嵌套状态直接修改 | passed | 决策和批注操作无效，提交导致刷新 |
| EnergyLoadScheduler | complex | passed | 是 | 模板隐藏响应式依赖 | passed | 状态文字变化但按钮颜色不更新 |
| ForecastSimulator | complex | failed | 是 | 初始化或生命周期迁移 | passed | 无初始模拟结果、图表和明细 |
| TopologyEditor | complex | execution_error | 是 | 嵌套状态直接修改 | passed | 标题、移动、连线和模拟日志异常 |
| DocumentApprovalFlow | complex | passed | 否 | 未发现差异 | passed | 人工浏览器检查未发现差异 |
| CapacityPlanningBoard | complex | failed | 是 | 模板语法错误 | passed | 错误的 s-for/trackby 导致解析失败 |
| GradebookMatrix | complex | failed | 是 | DOM 子节点顺序 | passed | 复选框和百分号位置不一致 |
| SupportTicketConsole | complex | failed | 是 | DOM 子节点顺序 | passed | 内部备注复选框位置不一致 |
| ExperimentRolloutConsole | complex | failed | 是 | 模板隐藏响应式依赖 | passed | 胜出版本行颜色不实时更新 |

## 5. 互斥主类别统计

| 主类别 | 样本数 | 占 14 个复核样本 |
| --- | ---: | ---: |
| 模板隐藏响应式依赖 | 3 | 21.43% |
| 未发现差异 | 3 | 21.43% |
| DOM 子节点顺序 | 2 | 14.29% |
| 初始化或生命周期迁移 | 2 | 14.29% |
| 嵌套状态直接修改 | 2 | 14.29% |
| 循环作用域绑定 | 1 | 7.14% |
| 模板语法错误 | 1 | 7.14% |

主类别用于论文中的互斥百分比统计；复合缺陷仍保存在 `sample_results.csv` 的 `multi_labels` 字段中。该 CSV 还保留完整人工现象、根因、修复内容、v1 错误码、证据路径以及修复前后 SHA-256，可用于论文案例追溯。

In [ ]:
multi_counts = summary['formal_30_v2']['multi_label_counts']
for category, count in sorted(multi_counts.items(), key=lambda item: (-item[1], item[0])):
    print(f'{category}: {count}')

## 6. 论文可用结论与限制

1. 独立测试集自动基线为 16/30，即 53.33%；若只统计正常完成样本则为 55.17%。
2. Repair 将 13 个初始失败样本转为自动通过，说明迭代修复有效，但代价为 1,767,836 个 Repair Token。
3. 自动通过率随复杂度显著下降，复杂组件是后续改进重点。
4. 14 个定向人工复核样本中，11 个需要修改代码；修复后 14 个均由用户在浏览器确认通过。
5. 响应式依赖、嵌套状态修改、初始化/生命周期和 DOM 顺序是主要运行时问题。
6. 自动评估不能替代真实浏览器检查，尤其无法稳定识别状态驱动的样式更新。
7. 基于这 14 个样本得到的 Prompt、Validator 或迁移规则改进属于事后开发；若要评价改进系统，必须使用新的未见测试集。

## 7. 派生文件

- `data/experiments/analysis/formal_30_v1_v2_20261022/summary.json`
- `data/experiments/analysis/formal_30_v1_v2_20261022/sample_results.csv`
- `data/experiments/analysis/formal_30_v1_v2_20261022/primary_defect_taxonomy.csv`
- `data/experiments/analysis/formal_30_v1_v2_20261022/REPORT.md`

重新运行第一个代码单元会先验证两个冻结快照，再重新生成上述派生文件。